# Module 4 Lab (Instructor Solution) — Three-Role Pipeline in LangGraph and CrewAI

**Researcher → Analyst → Critic**  
Estimated time: **100–130 minutes** · Learning outcomes: **LO 4.5 and LO 4.6**

This notebook replaces Lab A. You will specify, implement, run, and compare the same three-role workflow in two frameworks. The critic may send the work back to the analyst, but the workflow permits **at most two revision cycles**.

> **Make it yours.** The included topic is only a runnable example. Replace it with a complex question from your own academic or professional domain. A personally meaningful topic will produce a much stronger debrief and framework comparison.

### Evidence boundary

The base lab does **not** give the researcher a live web-search tool. Its output is a synthesis of model knowledge, not verified current research. Do not present generated source details as verified. For assessed factual work, give the pipeline a source packet or add an approved search/retrieval tool and verify every citation.


## How to use this notebook

1. In Google Colab, choose **Runtime → Run all** after completing each `TODO`.
2. Use the default **Hugging Face Inference Providers** route for the capable open-weight model (a Hugging Face token is required; free credits and quotas can change).
3. Alternatively choose OpenAI or Groq, or use Ollama locally without an API key.
4. Never paste a key directly into a saved code cell. The setup cell uses a masked prompt.
5. Run **both frameworks on exactly the same topic and model** for a fair comparison.
6. Before submission, keep outputs visible and rename the file to `Module4_Unit4_AltFramework_[YourName].ipynb`.

### Required notebook evidence

- Completed role-specification table (five fields for each role)
- LangGraph output and structured execution log
- CrewAI output and structured execution log
- Answers to all three Lab B debrief questions (at least two sentences each)
- A framework comparison of at least 200 words covering all four required dimensions


In [1]:
# Colab setup (usually 1–3 minutes). Restart the runtime only if Colab asks.
%pip install -q "langgraph>=1.0,<2" "litellm>=1.75,<2" \
    "crewai[litellm]>=1.15,<1.16"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.0/195.0 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 114.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.

In [2]:
import json
import logging
import os
import re
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
from typing import Literal, TypedDict

from IPython.display import Markdown, display
from langgraph.graph import END, START, StateGraph
from crewai import Agent, Crew, LLM, Process, Task
from litellm import completion

LOG_PATH = Path("module4_three_role_pipeline.log")
logger = logging.getLogger("module4_three_role_pipeline")
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter("%(message)s")
file_handler = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.propagate = False

EVENTS: list[dict] = []

def log_event(framework: str, role: str, event: str, **details) -> None:
    '''Write one JSON object per line, safe for later analysis.'''
    record = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "framework": framework,
        "role": role,
        "event": event,
        **details,
    }
    EVENTS.append(record)
    logger.info(json.dumps(record, ensure_ascii=False))
    print(f"[{framework} | {role}] {event}")

print("Imports and structured logging are ready.")


Imports and structured logging are ready.


## Provider and model configuration


In [3]:
# LiteLLM gives both implementations one open interface to many model providers.
# Choose: "huggingface" (default), "groq", "openai", or "ollama" (local/no key).
PROVIDER = "huggingface"

# The prefix before the first slash is the LiteLLM provider. For Hugging Face,
# "openai/" selects its OpenAI-compatible router; the remainder is the HF model ID.
MODEL_CONFIGS = {
    "huggingface": {
        "model": "openai/openai/gpt-oss-120b:cheapest",
        "display_model": "openai/gpt-oss-120b:cheapest",
        "api_base": "https://router.huggingface.co/v1",
        "key_env": "HF_TOKEN",
    },
    "groq": {
        "model": "groq/openai/gpt-oss-120b",
        "display_model": "openai/gpt-oss-120b",
        "key_env": "GROQ_API_KEY",
    },
    "openai": {
        "model": "openai/gpt-4.1-mini",
        "display_model": "gpt-4.1-mini",
        "key_env": "OPENAI_API_KEY",
    },
    "ollama": {
        "model": "ollama/qwen3:8b",
        "display_model": "qwen3:8b",
        "api_base": "http://localhost:11434",
        "key_env": None,
    },
}

if PROVIDER not in MODEL_CONFIGS:
    raise ValueError(f"Unknown provider: {PROVIDER}")
MODEL_CONFIG = MODEL_CONFIGS[PROVIDER]

def require_secret(env_name: str) -> str:
    if not os.getenv(env_name):
        os.environ[env_name] = getpass(f"Enter {env_name} (input is hidden): ")
    if not os.getenv(env_name):
        raise ValueError(f"{env_name} is required for the selected provider.")
    return os.environ[env_name]

def shared_llm_settings() -> dict:
    settings = {"model": MODEL_CONFIG["model"], "temperature": 0.2}
    if MODEL_CONFIG.get("api_base"):
        settings["api_base"] = MODEL_CONFIG["api_base"]
    if MODEL_CONFIG.get("key_env"):
        settings["api_key"] = require_secret(MODEL_CONFIG["key_env"])
    return settings

LITELLM_SETTINGS = shared_llm_settings()

def call_shared_llm(messages: list[dict]) -> str:
    '''One provider-neutral call path used by the LangGraph implementation.'''
    response = completion(messages=messages, **LITELLM_SETTINGS)
    content = response.choices[0].message.content
    return (content or "").strip()

def make_crewai_llm() -> LLM:
    # CrewAI uses `base_url`; LiteLLM's Python function uses `api_base`.
    settings = {
        "model": LITELLM_SETTINGS["model"],
        "temperature": LITELLM_SETTINGS["temperature"],
    }
    if "api_base" in LITELLM_SETTINGS:
        settings["base_url"] = LITELLM_SETTINGS["api_base"]
    if "api_key" in LITELLM_SETTINGS:
        settings["api_key"] = LITELLM_SETTINGS["api_key"]
    return LLM(**settings)

crewai_llm = make_crewai_llm()
print(f"Provider: {PROVIDER} | Model: {MODEL_CONFIG['display_model']}")
print("Shared model interface: LiteLLM")


Enter HF_TOKEN (input is hidden): ··········
Provider: huggingface | Model: openai/gpt-oss-120b:cheapest
Shared model interface: LiteLLM


## Choose one shared task


In [4]:
# Replace this example with a complex topic from your own domain.
EXAMPLE_TOPIC = (
    "What organizational policies could reduce burnout among distributed "
    "software teams without reducing productivity? Analyze trade-offs and "
    "make three actionable recommendations for a 200-person technology company."
)

# Strongly recommended: edit this value.
TOPIC = EXAMPLE_TOPIC

assert len(TOPIC.split()) >= 12, "Use a sufficiently complex, specific topic."
print("Pipeline topic:\n", TOPIC)


Pipeline topic:
 What organizational policies could reduce burnout among distributed software teams without reducing productivity? Analyze trade-offs and make three actionable recommendations for a 200-person technology company.


## 1. Role specification — complete before implementation

| Field | Researcher | Analyst | Critic |
|---|---|---|---|
| **Name + one-sentence responsibility** | Evidence Mapper: creates a bounded set of relevant evidence notes and uncertainties. | Decision Analyst: converts the evidence notes into a structured, decision-oriented analysis. | Quality Gatekeeper: evaluates reasoning plus deterministic format/evidence criteria and returns an explicit verdict. |
| **Input format** | `task: str` containing a specific professional question. | `task: str`, `research_output: str` with `[R#]` evidence markers, plus optional `critic_feedback: str`. | `analysis_output: str` and a deterministic-check dictionary. |
| **Output format** | Six concise evidence notes labeled `[R1]`–`[R6]`, followed by `Uncertainties and verification needs`. | A 400–700 word Markdown report with four required headings and at least three distinct `[R#]` markers. | First line exactly `APPROVED` or `REVISE`, followed by criterion-specific feedback. |
| **Handoff condition** | At least six relevant notes are present, claims are scoped, and uncertainties are identified. | All four headings are present, length is 400–700 words, and at least three research markers are used. | Approve only if deterministic checks pass and no material reasoning defect is found; otherwise return actionable feedback. |
| **Position-specific failure + structural mitigation** | Unverified model knowledge can look like retrieved evidence; label notes as unverified and preserve an uncertainty section in state. | Revision can silently ignore critic feedback; pass feedback explicitly and retain every version in `analysis_history`. | Always-reject behavior can loop forever; route with a hard maximum of two completed revisions and record the stop reason. |

Adapt these specifications to your topic. The structural checks below intentionally implement the Analyst and Critic handoff conditions.


In [5]:
## Shared, deterministic quality gate
REQUIRED_HEADINGS = (
    "## Executive Summary",
    "## Evidence-Based Analysis",
    "## Recommendations",
    "## Limitations",
)
MIN_WORDS = 400
MAX_WORDS = 700
MIN_EVIDENCE_MARKERS = 3
MAX_REVISIONS = 2

def structural_checks(text: str) -> dict:
    """Return observable pass/fail facts; the LLM critic handles reasoning quality."""
    word_count = len(re.findall(r"\b\w+[\w'-]*\b", text))
    markers = sorted(set(re.findall(r"\[R\d+\]", text)))
    missing = [heading for heading in REQUIRED_HEADINGS if heading not in text]
    return {
        "word_count": word_count,
        "word_count_ok": MIN_WORDS <= word_count <= MAX_WORDS,
        "evidence_markers": markers,
        "evidence_count_ok": len(markers) >= MIN_EVIDENCE_MARKERS,
        "missing_headings": missing,
        "headings_ok": not missing,
    }

def structure_passes(checks: dict) -> bool:
    return all(checks[key] for key in ("word_count_ok", "evidence_count_ok", "headings_ok"))

def first_line_verdict(text: str) -> Literal["APPROVED", "REVISE"]:
    first = text.strip().splitlines()[0].strip().upper() if text.strip() else ""
    return "APPROVED" if first == "APPROVED" else "REVISE"

print({"criteria": {"words": [MIN_WORDS, MAX_WORDS], "minimum_markers": MIN_EVIDENCE_MARKERS,
                    "required_headings": REQUIRED_HEADINGS}, "max_revisions": MAX_REVISIONS})


{'criteria': {'words': [400, 700], 'minimum_markers': 3, 'required_headings': ('## Executive Summary', '## Evidence-Based Analysis', '## Recommendations', '## Limitations')}, 'max_revisions': 2}


## 2. LangGraph implementation

LangGraph makes state and routing explicit. `revision_count` means **completed analyst revisions**, not critic rejections: 0 is the initial analysis, 1 and 2 are revisions. This definition makes “maximum two revision cycles” unambiguous.


In [6]:
class PipelineState(TypedDict):
    task: str
    research_output: str
    analysis_output: str
    critic_verdict: str
    critic_feedback: str
    revision_count: int
    analysis_history: list[str]
    final_output: str
    stop_reason: str

def call_langgraph_role(system_prompt: str, user_prompt: str) -> str:
    return call_shared_llm([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ])


In [7]:
RESEARCHER_SYSTEM = """You are the Evidence Mapper. Produce exactly six concise, relevant
evidence notes labeled [R1] through [R6], then a section titled 'Uncertainties and
verification needs'. Distinguish established concepts from assumptions. You have no live
search tool: never claim that you browsed, and never invent URLs, quotations, or precise
bibliographic details. These are candidate evidence notes for later verification."""

ANALYST_SYSTEM = """You are the Decision Analyst. Write a 400–700 word decision-oriented
Markdown report using exactly these headings: ## Executive Summary, ## Evidence-Based
Analysis, ## Recommendations, and ## Limitations. Use at least three distinct [R#] markers
to show which research notes support claims. Give three numbered, actionable recommendations.
Address trade-offs and do not add facts or citations absent from the research notes."""

CRITIC_SYSTEM = """You are the Quality Gatekeeper. Evaluate analytical reasoning,
traceability to the supplied research, actionable recommendations, trade-offs, and candid
limitations. The deterministic checks are binding. First line must be exactly APPROVED or
REVISE. Approve only when every deterministic check passes and there is no material reasoning
defect. After the first line, briefly explain the decision and give actionable feedback."""

def researcher_node(state: PipelineState) -> dict:
    log_event("LangGraph", "Researcher", "started")
    output = call_langgraph_role(RESEARCHER_SYSTEM, f"Professional question:\n{state['task']}")
    log_event("LangGraph", "Researcher", "completed", output_chars=len(output))
    return {"research_output": output}

def analyst_node(state: PipelineState) -> dict:
    is_revision = bool(state.get("critic_feedback"))
    revision_count = state.get("revision_count", 0) + (1 if is_revision else 0)
    prompt = f"""Question:\n{state['task']}\n\nResearch notes:\n{state['research_output']}

This is {'revision ' + str(revision_count) if is_revision else 'the initial analysis'}.
Critic feedback to address:\n{state.get('critic_feedback') or 'None — initial draft.'}"""
    log_event("LangGraph", "Analyst", "started", revision_count=revision_count)
    output = call_langgraph_role(ANALYST_SYSTEM, prompt)
    history = [*state.get("analysis_history", []), output]
    log_event("LangGraph", "Analyst", "completed", revision_count=revision_count,
              output_chars=len(output))
    return {
        "analysis_output": output,
        "analysis_history": history,
        "revision_count": revision_count,
        "final_output": output,
        "critic_feedback": "",  # consume feedback so a later call counts only once
    }

def critic_node(state: PipelineState) -> dict:
    checks = structural_checks(state["analysis_output"])
    prompt = f"""Question:\n{state['task']}\n\nResearch notes:\n{state['research_output']}

Analysis to evaluate:\n{state['analysis_output']}\n\nDeterministic checks:\n{json.dumps(checks, indent=2)}"""
    log_event("LangGraph", "Critic", "started", revision_count=state["revision_count"],
              structural_checks=checks)
    critique = call_langgraph_role(CRITIC_SYSTEM, prompt)
    model_verdict = first_line_verdict(critique)
    verdict = "APPROVED" if structure_passes(checks) and model_verdict == "APPROVED" else "REVISE"
    feedback = critique if verdict == "REVISE" else ""
    log_event("LangGraph", "Critic", "completed", verdict=verdict,
              revision_count=state["revision_count"])
    return {"critic_verdict": verdict, "critic_feedback": feedback}

def critic_routing(state: PipelineState) -> Literal["analyst", "finalize"]:
    if state["critic_verdict"] == "APPROVED":
        return "finalize"
    if state["revision_count"] >= MAX_REVISIONS:
        return "finalize"
    return "analyst"

def finalize_node(state: PipelineState) -> dict:
    approved = state["critic_verdict"] == "APPROVED"
    reason = "approved" if approved else f"hard stop after {state['revision_count']} revisions"
    log_event("LangGraph", "Pipeline", "finished", stop_reason=reason,
              total_analysis_versions=len(state["analysis_history"]))
    return {"final_output": state["analysis_output"], "stop_reason": reason}


In [8]:
builder = StateGraph(PipelineState)
builder.add_node("researcher", researcher_node)
builder.add_node("analyst", analyst_node)
builder.add_node("critic", critic_node)
builder.add_node("finalize", finalize_node)

builder.add_edge(START, "researcher")
builder.add_edge("researcher", "analyst")
builder.add_edge("analyst", "critic")
builder.add_conditional_edges(
    "critic",
    critic_routing,
    {"analyst": "analyst", "finalize": "finalize"},
)
builder.add_edge("finalize", END)
langgraph_pipeline = builder.compile()

display(Image(langgraph_pipeline.get_graph().draw_mermaid_png())) if False else print(
    "Graph compiled: START → researcher → analyst → critic ⇄ analyst → finalize → END"
)


Graph compiled: START → researcher → analyst → critic ⇄ analyst → finalize → END


In [ ]:
initial_state: PipelineState = {
    "task": TOPIC,
    "research_output": "",
    "analysis_output": "",
    "critic_verdict": "",
    "critic_feedback": "",
    "revision_count": 0,
    "analysis_history": [],
    "final_output": "",
    "stop_reason": "",
}

langgraph_result = langgraph_pipeline.invoke(initial_state, {"recursion_limit": 12})
display(Markdown("### LangGraph final output\n\n" + langgraph_result["final_output"]))
print("\nStop reason:", langgraph_result["stop_reason"])
print("Analysis versions:", len(langgraph_result["analysis_history"]))


[LangGraph | Researcher] started
[LangGraph | Researcher] completed
[LangGraph | Analyst] started
[LangGraph | Analyst] completed
[LangGraph | Critic] started
[LangGraph | Critic] completed
[LangGraph | Analyst] started


### LangGraph execution debrief — write at least two sentences per answer

1. **Which role-specification decision had the largest behavioral impact? What evidence in the log supports your answer?**  
   TODO

2. **If you added a fourth agent, what would it do, where would it go, and what new coordination failure would it introduce?**  
   TODO

3. **How does this state schema compare with a simpler one-agent schema? What does its field count reveal about agent count and state complexity?**  
   TODO


## 3. Equivalent CrewAI implementation

CrewAI's `Agent`, `Task`, and `Crew` abstractions handle each role invocation. A small bounded Python loop makes the critic-triggered revision behavior directly comparable with LangGraph. Each stage is intentionally run as a one-task sequential Crew so the exact handoff text can be logged and inspected.


In [ ]:
researcher = Agent(
    role="Researcher",
    goal="Create bounded, relevant evidence notes for the supplied professional question",
    backstory="An evidence mapper who distinguishes useful knowledge from uncertainty.",
    llm=crewai_llm,
    allow_delegation=False,
    max_iter=3,
    verbose=True,
)
analyst = Agent(
    role="Analyst",
    goal="Produce a structured, evidence-traceable decision analysis",
    backstory="A systematic analyst who makes trade-offs and assumptions explicit.",
    llm=crewai_llm,
    allow_delegation=False,
    max_iter=3,
    verbose=True,
)
critic = Agent(
    role="Critic",
    goal="Apply an exacting but bounded quality gate and provide actionable feedback",
    backstory="A fair quality controller who approves work that meets explicit criteria.",
    llm=crewai_llm,
    allow_delegation=False,
    max_iter=3,
    verbose=True,
)

async def run_crewai_task(agent: Agent, description: str, expected_output: str, role: str) -> str:
    log_event("CrewAI", role, "started", input_chars=len(description))
    task = Task(description=description, expected_output=expected_output, agent=agent)
    crew = Crew(
      agents=[agent],
      tasks=[task],
      process=Process.sequential,
      verbose=True
    )
    result = await crew.kickoff_async()
    output = str(result).strip()
    log_event("CrewAI", role, "completed", output_chars=len(output))
    return output

async def run_crewai_pipeline(topic: str, max_revisions: int = MAX_REVISIONS) -> dict:
    research = await run_crewai_task(
        researcher,
        RESEARCHER_SYSTEM + f"\n\nProfessional question:\n{topic}",
        "Six labeled evidence notes [R1]–[R6], then uncertainties and verification needs.",
        "Researcher",
    )
    history: list[str] = []
    feedback = ""
    revision_count = 0
    while True:
        analysis = await run_crewai_task(
            analyst,
            ANALYST_SYSTEM + f"""\n\nQuestion:\n{topic}\n\nResearch notes:\n{research}

This is {'revision ' + str(revision_count) if revision_count else 'the initial analysis'}.
Critic feedback to address:\n{feedback or 'None — initial draft.'}""",
            "A 400–700 word report with all required headings, three recommendations, and [R#] markers.",
            "Analyst",
        )
        history.append(analysis)
        checks = structural_checks(analysis)
        critique = await run_crewai_task(
            critic,
            CRITIC_SYSTEM + f"""\n\nQuestion:\n{topic}\n\nResearch notes:\n{research}

Analysis to evaluate:\n{analysis}\n\nDeterministic checks:\n{json.dumps(checks, indent=2)}""",
            "First line exactly APPROVED or REVISE, followed by criterion-specific reasoning.",
            "Critic",
        )
        model_verdict = first_line_verdict(critique)
        verdict = "APPROVED" if structure_passes(checks) and model_verdict == "APPROVED" else "REVISE"
        log_event("CrewAI", "Router", "decision", verdict=verdict,
                  revision_count=revision_count, structural_checks=checks)
        if verdict == "APPROVED":
            stop_reason = "approved"
            break
        if revision_count >= max_revisions:
            stop_reason = f"hard stop after {revision_count} revisions"
            break
        revision_count += 1
        feedback = critique

    log_event("CrewAI", "Pipeline", "finished", stop_reason=stop_reason,
              total_analysis_versions=len(history))
    return {
        "task": topic,
        "research_output": research,
        "analysis_output": analysis,
        "critic_verdict": verdict,
        "critic_feedback": critique,
        "revision_count": revision_count,
        "analysis_history": history,
        "final_output": analysis,
        "stop_reason": stop_reason,
    }


In [ ]:
import nest_asyncio
nest_asyncio.apply()

crewai_result = await run_crewai_pipeline(TOPIC)
display(Markdown("### CrewAI final output\n\n" + crewai_result["final_output"]))
print("\nStop reason:", crewai_result["stop_reason"])
print("Analysis versions:", len(crewai_result["analysis_history"]))

In [ ]:
# Inspect a concise, framework-by-role execution trace.
for event in EVENTS:
    print(event["timestamp_utc"], event["framework"], event["role"], event["event"],
          event.get("verdict", ""), event.get("revision_count", ""))

print(f"\nFull JSONL log saved to: {LOG_PATH.resolve()}")


## 4. Framework Comparison Report — minimum 200 words

Use evidence from **your own code and execution**. Count only implementation lines (exclude imports, blank lines, comments, prompts, display code, and the shared quality-gate helper) and state your counting rule.

| Dimension | LangGraph evidence | CrewAI evidence |
|---|---|---|
| Implementation verbosity | TODO: measured LOC + configuration details | TODO: measured LOC + configuration details |
| Observability | TODO: cite specific state/log records | TODO: cite specific task/crew/log records |
| Customization flexibility | TODO: name exact routing/state edits | TODO: name exact loop/task edits |
| Developer experience | TODO: identify a specific knowledge gap | TODO: identify a specific knowledge gap |

### Structured report (≥200 words)

**Implementation verbosity.** TODO

**Observability.** TODO

**Customization flexibility.** TODO

**Developer experience.** TODO


## Offline contract checks


In [ ]:
# Fast offline checks of the deterministic gate and routing contract (no LLM calls).
valid_sample = "\n".join([
    "## Executive Summary", "[R1] " + ("summary " * 95),
    "## Evidence-Based Analysis", "[R2] " + ("analysis " * 105),
    "## Recommendations", "1. Act [R3]. " + ("recommendation " * 105),
    "## Limitations", "limits " * 105,
])
valid_checks = structural_checks(valid_sample)
assert structure_passes(valid_checks), valid_checks
assert first_line_verdict("APPROVED\nMeets criteria") == "APPROVED"
assert first_line_verdict("APPROVED with reservations") == "REVISE"
assert critic_routing({"critic_verdict": "APPROVED", "revision_count": 0}) == "finalize"
assert critic_routing({"critic_verdict": "REVISE", "revision_count": 1}) == "analyst"
assert critic_routing({"critic_verdict": "REVISE", "revision_count": 2}) == "finalize"
print("All offline quality-gate and routing checks passed.")


## 5. Submission checklist and troubleshooting

- [ ] I replaced the example topic (or explained why it is professionally relevant to me).
- [ ] All role-specification fields are complete.
- [ ] Both frameworks used the same topic, provider, model, and quality criteria.
- [ ] Both executions and outputs are visible.
- [ ] The log shows every role invocation and any revisions.
- [ ] Each debrief response has at least two sentences.
- [ ] The comparison is at least 200 words and uses measured evidence.
- [ ] I reviewed model-generated claims and did not describe unverified notes as live research.
- [ ] I renamed and downloaded the notebook.

**Common issues**

- `401`/authentication error: rerun the provider cell and enter the correct provider token.
- Rate limit: wait for the provider window to reset or select another provider/model.
- Model not found: providers retire model IDs; choose a current model and update `MODEL_CONFIGS`.
- Hugging Face routing error: confirm the token permits Inference Providers and try another routing suffix or supported model.
- Ollama connection error: Ollama must be running and `ollama pull qwen3:8b` must complete first.
- The critic never approves: inspect deterministic checks and the feedback; the hard limit still terminates safely.

**Current documentation used when this template was authored (August 2026):**
[LangGraph Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api) ·
[CrewAI LLM connections](https://docs.crewai.com/en/learn/llm-connections) ·
[Hugging Face Inference Providers](https://huggingface.co/docs/inference-providers/en/index) ·
[LiteLLM providers](https://docs.litellm.ai/)
